# 第 5 章：魔术方法与运算符重载

> 本章目标：掌握以双下划线包围的**魔术方法（magic methods / dunder methods）**，让自定义类支持 `print`、`+`、`==`、`[]`、`for`、`with` 等内置操作--像内置类型一样自然。

---

## 5.1 魔术方法全景图

魔术方法是 Python 留给类的"**协议接口**"：解释器在执行 `+`、`print`、`for` 等操作时，会自动回调这些方法。

```mermaid
mindmap
  root((魔术方法))
    对象创建与销毁
      __new__
      __init__
      __del__
    字符串表示
      __repr__
      __str__
      __format__
    比较运算
      __eq__ __ne__
      __lt__ __gt__
      __hash__
    算术运算
      __add__ __sub__
      __radd__ 反向
      __iadd__ 就地
    容器与迭代
      __len__
      __getitem__
      __iter__
      __contains__
    可调用与上下文
      __call__
      __enter__ __exit__
```

> ⚠️ **警告**：永远不要自己定义新的双下划线方法名--那是 Python 留给自己的命名空间。

## 5.2 `__str__` vs `__repr__`：给对象一张"名片"

| 对比 | `__str__` | `__repr__` |
|------|-----------|------------|
| 目标读者 | **用户**（终端、日志） | **开发者**（调试、交互式环境） |
| 要求 | 可读性好 | **无歧义**，理想情况下 `eval(repr(obj))` 能还原对象 |
| 触发场景 | `print(obj)`、`str(obj)`、f-string | 交互式回显、容器内的元素、`repr(obj)` |
| 只定义一个时 | 回退到 `__repr__` | 不会回退到 `__str__` |

```mermaid
flowchart LR
    A[print obj] --> B{有 __str__?}
    B -- 有 --> C[调用 __str__]
    B -- 没有 --> D[调用 __repr__]
    E[交互式回显 / 容器打印] --> F[调用 __repr__]
```

In [1]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Point(x={self.x!r}, y={self.y!r})"   # !r 会调用元素的 __repr__

    def __str__(self):
        return f"({self.x}, {self.y})"


p = Point(3, 4)
print(p)           # __str__ -> (3, 4)
print(str(p))      # __str__
print(repr(p))     # __repr__ -> Point(x=3, y=4)

# 容器中的元素打印时用 __repr__（即使 __str__ 存在）
print([p, Point(0, 0)])
print(f"{p!r}")   # f-string 中 !r 强制用 repr

(3, 4)
(3, 4)
Point(x=3, y=4)
[Point(x=3, y=4), Point(x=0, y=0)]
Point(x=3, y=4)


## 5.3 比较运算：`__eq__`、`__hash__` 与 `total_ordering`
默认情况下，`p1 == p2` 比较的是**身份**（是否同一个对象，等价于 `is`）。重写 `__eq__` 后按**值**比较。

⚠️ **铁律**：重写 `__eq__` 会自动把 `__hash__` 置为 `None`（对象不可哈希，放进 set/dict 会报错）。若要可哈希，需同时实现 `__hash__`：**相等的对象必须有相同的哈希值**。

In [2]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __eq__(self, other):
        if not isinstance(other, Point):
            return NotImplemented   # 告诉解释器：我不会比这种类型，去问对方
        return (self.x, self.y) == (other.x, other.y)

    def __hash__(self):
        return hash((self.x, self.y))   # 基于参与相等判断的字段


p1, p2 = Point(1, 2), Point(1, 2)
print(p1 == p2)        # True：按值比较
print(p1 is p2)        # False：不是同一个对象
print(p1 == "字符串")  # False：走 NotImplemented 分支后回退

s = {p1, p2}           # 可哈希 -> set 中只留一个
print(len(s))          # 1
d = {p1: "点A"}
print(d[Point(1, 2)])  # "点A"：hash 相同且相等 -> 找到

True
False
False
1
点A


### 一次实现全部比较：`functools.total_ordering`

手写 `<`、`<=`、`>`、`>=` 四个方法既啰嗦又容易写出不一致的逻辑。`total_ordering` 只要求你实现 `__eq__` + 一个排序方法，其余自动推导：

In [3]:
from functools import total_ordering


@total_ordering
class Version:
    def __init__(self, major, minor):
        self.major, self.minor = major, minor

    def __eq__(self, other):
        return (self.major, self.minor) == (other.major, other.minor)

    def __lt__(self, other):           # 只写 "小于"，其余自动推导
        return (self.major, self.minor) < (other.major, other.minor)

    def __repr__(self):
        return f"v{self.major}.{self.minor}"


v1, v2, v3 = Version(1, 2), Version(2, 0), Version(1, 2)
print(v1 < v2, v1 <= v3, v2 > v1, v1 >= v3)   # True True True True
print(sorted([v3, v2, v1]))                    # 排序也直接可用

True True True True
[v1.2, v1.2, v2.0]


## 5.4 算术运算符重载
让自定义类型支持 `+`、`-`、`*` 等。以向量加法为例：

```mermaid
flowchart TD
    A["v1 + v2"] --> B["尝试 v1.__add__(v2)"]
    B -- 返回值 --> R[得到结果]
    B -- NotImplemented --> C["尝试 v2.__radd__(v1)"]
    C -- 返回值 --> R
    C -- NotImplemented --> D[TypeError]
```

- `__radd__`：**反向运算**，当左操作数不认识右操作数时（如 `3 + v`，int 的 `__add__` 返回 NotImplemented）由右操作数接手；
- `__iadd__`：**就地运算**，对应 `+=`（不定义时会回退到 `__add__`）。

In [4]:
class Vector:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __repr__(self):
        return f"Vector({self.x}, {self.y})"

    def __add__(self, other):
        if isinstance(other, Vector):
            return Vector(self.x + other.x, self.y + other.y)
        return NotImplemented

    def __radd__(self, other):        # 支持 3 + v
        if isinstance(other, (int, float)):
            return Vector(self.x + other, self.y + other)
        return NotImplemented

    def __mul__(self, scalar):        # v * 3
        if isinstance(scalar, (int, float)):
            return Vector(self.x * scalar, self.y * scalar)
        return NotImplemented

    __rmul__ = __mul__                # 3 * v 走同一段逻辑

    def __neg__(self):                # -v
        return Vector(-self.x, -self.y)

    def __abs__(self):                # abs(v) -> 模长
        return (self.x ** 2 + self.y ** 2) ** 0.5


v = Vector(1, 2)
print(v + Vector(3, 4))   # Vector(4, 6)
print(v * 3)              # Vector(3, 6)
print(3 * v)              # Vector(3, 6)  ← __rmul__
print(10 + v)             # Vector(11, 12) ← __radd__
print(-v)                 # Vector(-1, -2)
print(abs(Vector(3, 4)))  # 5.0

Vector(4, 6)
Vector(3, 6)
Vector(3, 6)
Vector(11, 12)
Vector(-1, -2)
5.0


## 5.5 容器协议：让类像 list/dict 一样工作

| 方法 | 对应操作 | 说明 |
|------|---------|------|
| `__len__` | `len(obj)` | 元素个数 |
| `__getitem__` | `obj[key]` | 取值（支持切片就看你怎么实现） |
| `__setitem__` | `obj[key] = v` | 赋值 |
| `__delitem__` | `del obj[key]` | 删除 |
| `__contains__` | `x in obj` | 不定义时回退到 `__iter__` 遍历 |
| `__iter__` | `for x in obj` | 返回迭代器 |

实现一个**类似扑克牌组**的类，感受"协议"的威力：只实现 `__len__` 和 `__getitem__`，就能免费获得迭代、`in`、切片、`random.choice` 等一堆能力！

In [5]:
import random

class FrenchDeck:
    ranks = [str(n) for n in range(2, 11)] + list("JQKA")
    suits = "♠ ♥ ♦ ♣".split()

    def __init__(self):
        self._cards = [f"{r}{s}" for s in self.suits for r in self.ranks]

    def __len__(self):
        return len(self._cards)

    def __getitem__(self, index):
        return self._cards[index]     # 直接转发，天然支持负索引和切片


deck = FrenchDeck()
print(len(deck))            # __len__ -> 52
print(deck[0], deck[-1])    # __getitem__ -> 2♠ A♣

# 没写 __iter__，但 __getitem__ 存在 -> 解释器用旧式迭代协议
for card in deck[:5]:
    print(card, end=" ")
print()

print("K♥" in deck)          # __contains__ 回退到迭代
print(random.choice(deck))   # 标准库函数直接可用！
print(sorted(deck)[:3])      # 排序也能用（按字符串序）

52
2♠ A♣
2♠ 3♠ 4♠ 5♠ 6♠ 
True
A♠
['10♠', '10♣', '10♥']


## 5.6 可迭代与迭代器：`__iter__` 与 `__next__`

两个容易混淆的概念：

- **可迭代对象（Iterable）**：实现了 `__iter__`，能被 for 循环（list、str、dict……）；
- **迭代器（Iterator）**：实现了 `__iter__` **和** `__next__`，是"取数的游标"。

```mermaid
flowchart LR
    A[for x in obj] --> B["iter(obj) 调用 obj.__iter__"]
    B --> C[得到迭代器 it]
    C --> D["next(it) 调用 it.__next__"]
    D --> E{还有元素?}
    E -- 有 --> F[yield x 给循环体]
    F --> D
    E -- 没有 --> G[StopIteration 结束循环]
```

In [6]:
class Countdown:
    """可迭代对象：每次 for 循环都返回一个全新的迭代器"""

    def __init__(self, start):
        self.start = start

    def __iter__(self):
        return CountdownIterator(self.start)   # 新游标


class CountdownIterator:
    """迭代器：真正负责逐个取值"""

    def __init__(self, current):
        self.current = current

    def __iter__(self):
        return self    # 迭代器的 __iter__ 返回自身

    def __next__(self):
        if self.current <= 0:
            raise StopIteration   # 结束信号
        self.current -= 1
        return self.current + 1


cd = Countdown(3)
print(list(cd))    # [3, 2, 1]
print(list(cd))    # [3, 2, 1] —— 可迭代对象可以被反复遍历

it = iter(cd)      # 手动驱动一次
print(next(it), next(it), next(it))
try:
    next(it)       # 耗尽后抛 StopIteration
except StopIteration:
    print("迭代器耗尽")

[3, 2, 1]
[3, 2, 1]
3 2 1
迭代器耗尽


## 5.7 上下文管理器：`__enter__` / `__exit__`

`with` 语句的背后协议--**保证资源无论如何都会被释放**：

```mermaid
flowchart TD
    A[with open f as fp] --> B["fp.__enter__()"]
    B --> C[执行 with 代码块]
    C -- 正常结束 --> D["fp.__exit__(None, None, None)"]
    C -- 抛异常 --> E["fp.__exit__(exc_type, exc_value, tb)"]
    E --> F{返回 True?}
    F -- 是 --> G[异常被吞掉]
    F -- 否 --> H[异常继续向外抛]
    D --> I[资源已清理]
    H --> I
```

`__exit__` 的三个参数在无异常时全是 `None`；有异常时分别是**异常类型、异常值、回溯对象**。

In [7]:
class Timer:
    """计时上下文管理器"""

    def __enter__(self):
        import time
        self.start = time.perf_counter()
        return self    # as 后面拿到的就是这个返回值

    def __exit__(self, exc_type, exc_value, traceback):
        import time
        self.elapsed = time.perf_counter() - self.start
        if exc_type is None:
            print(f"正常结束，耗时 {self.elapsed:.4f} 秒")
        else:
            print(f"发生异常 {exc_type.__name__}，但资源照样清理")
        return False    # False = 不吞异常，让它继续抛


with Timer():
    total = sum(range(1_000_000))

try:
    with Timer():
        raise ValueError("业务出错")
except ValueError as e:
    print(f"异常穿透: {e}")   # __exit__ 返回 False，异常照常抛出

正常结束，耗时 0.0246 秒
发生异常 ValueError，但资源照样清理
异常穿透: 业务出错


## 5.8 `__call__`：让实例像函数一样被调用

实现 `__call__` 的对象叫**可调用对象（callable）**--状态和逻辑封装在一起，比闭包更清晰：

In [8]:
class Counter:
    def __init__(self):
        self.count = 0

    def __call__(self, step=1):
        self.count += step
        return self.count


c = Counter()
print(callable(c))   # True
print(c(), c(), c(10))   # 像函数一样调用 1 2 12
print(c.count)       # 状态保留在实例上

True
1 2 12
12


## 5.9 本章小结

| 想支持的操作 | 实现的魔术方法 |
|-------------|---------------|
| `print` 好看 | `__str__` / `__repr__` |
| `==`、可放进 set | `__eq__` + `__hash__` |
| `<` 等全套比较 | `__lt__` + `@total_ordering` |
| `+` `-` `*` | `__add__` `__sub__` `__mul__`（配 `__radd__` 等） |
| `len()` / `[]` | `__len__` / `__getitem__` |
| for 循环 | `__iter__`（迭代器再加 `__next__`） |
| `with` | `__enter__` / `__exit__` |
| `obj()` | `__call__` |

> 💡 设计原则：**不要为了炫技重载运算符**。只有当操作在领域上自然成立时才重载--向量相加 ✅，两支股票相加 ❌。

### 📝 动手练习

1. 实现 `Money` 类：支持 `Money(10, 5) + Money(3, 2)`（元、角自动进位）、`==`、`<`、`repr`。
2. 给 `FrenchDeck` 加上 `__setitem__`，实现 `random.shuffle(deck)` 洗牌（提示：shuffle 依赖 `__setitem__`）。
3. 写一个 `DatabaseConnection` 上下文管理器模拟类：`__enter__` 打印"连接中"，`__exit__` 打印"连接已关闭"。

---
**下一章** 👉 `06_类方法_静态方法与抽象基类.ipynb`：厘清三种方法的边界，学习用 ABC 规范接口设计。